### Importar modulos necessários para acessar os codigos

In [1]:
from Treinos.Treinamento_Unet import treinar_modelo
from Testes.Teste_Unet import avaliar_e_visualizar
from Modelos.UNet_baseline import UNetBaseline
from Modelos.UNet_depth_reduced import UNetDepthReduced
from Modelos.UNet_MobileNet_v2 import UNetMobileNetV2
from Modelos.UNet_MobileNet_v3 import UNetMobileNetV3
from Modelos.UNet_SkipConnections import UNetElementWise
from Modelos.UNet_AttentionGates import UNetAttentionGates
from Modelos.UNet_PSA import UNetPSA
from Modelos.UNet_ResNet34 import UNetResNet34
from Modelos.UNet_SegFormer import SegFormerB0
from Modelos.UNet_MobileNetV3_AttentionGates import UNetMobileNetV3AttentionGates
from Modelos.UNet_MobileNetV3_AttentionGates_DPU import UNetMobileNetV3AttentionGatesDPU
from Modelos.UNet_MobileNet_v3_dpu import UNetMobileNetV3_dpu
from Modelos.HyperStarcop_oficial import carregar_hyperstarcop
from Modelos.UNet_ResNet34_dpu import UNetResNet34 as UNetResNet34_dpu
from Modelos.UNet_SegFormer_DPU import SegFormerB0_DPU
from Modelos.UNet_SegFormer_AttentionGates import SegFormerB0_Attention
from Modelos.UNet_SegFormer_AttentionGates_DPU import SegFormerB0_Attention_DPU

/media/jacques/hdd/Laboratorio/Projeto_joao/Methane_segmentation/.venv_hgq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Selecionar modelo

#### Descomente na lista MODELS os modelos que vai treinar. Cada item contém classe, nome e starting_point.


In [2]:
def HyperSTARCOP(in_channels=4, out_channels=1):
    return carregar_hyperstarcop(
        "Modelos_treinados/HyperSTARCOP_oficial/final_checkpoint_model.ckpt",
        "cpu",
    )

MODELS = [#(UNetBaseline, "UNet_baseline", 0),
          #(UNetMobileNetV2, "Mobile_Net_v2", 0),
          #(UNetMobileNetV3, "MobileNet_v3", 0),
          #(UNetMobileNetV3AttentionGates, "MobileNetV3_AttentionGates", 0),
          #(SegFormerB0, "UNet_SegFormer", 0),
          #(SegFormerB0_Attention, "UNet_SegFormer_AttGate", 0),
          (SegFormerB0_DPU, "UNet_SegFormer_DPU", 0),
          #(SegFormerB0_Attention_DPU, "UNet_SegFormer_AttGate_DPU", 0),
          ]


Treinar  Mag1c + RGB

In [3]:
# Escolha da função de perda: "BCE" ou "FD" (FocalDice)
selected_loss = "FD"
print("Loss selecionada:", selected_loss)

Loss selecionada: FD


In [ ]:
for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_" + selected_loss + "_mag1c_rgb"
    treinar_modelo(MODELO, nome_modelo, starting_point, selected_loss=selected_loss, produtos_entrada=["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"])

### Treinar quantizado (QAT com HGQ2, para hls4ml)

Não substitui o treino normal acima. Usa `selected_loss` e o mesmo split/dados, mas treina do zero já quantizado e salva `Modelos_treinados/<nome>.keras`.
Precisa de `keras>=3.15`, `HGQ2` e `hls4ml` (fora do `.venv` atual; use um venv separado).

In [4]:
import os
os.environ.setdefault("KERAS_BACKEND", "torch")  # antes de importar keras

from Treinos.Treinamento_unet_quantizado import treinar_modelo_quantizado

# "int8" (8 bits fixos) ou "hgq" (bits treináveis + EBOPs)
MODO_QUANT = "hgq"

MODELS_QUANT = [(SegFormerB0_DPU, "UNet_SegFormer_DPU", 0.0),
                #(SegFormerB0_Attention_DPU, "UNet_SegFormer_AttGate_DPU", 0.0),
                ]

for MODELO, NOME_MODELO, starting_point in MODELS_QUANT:
    nome_modelo = f"{NOME_MODELO}_HGQ_{MODO_QUANT}_{selected_loss}_mag1c_rgb"
    treinar_modelo_quantizado(MODELO, nome_modelo, starting_point, selected_loss=selected_loss,
                              produtos_entrada=["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"],
                              modo=MODO_QUANT)


--- Iniciando Treinamento Quantizado (hgq): UNet_SegFormer_DPU_HGQ_hgq_FD_mag1c_rgb ---
Dispositivo: cuda
Verificação concluída! 3425 amostras válidas selecionadas após verificar 3425 de 3425 linhas.
Amostras Treino: 2911 | Amostras Validação: 514
[ConversorHGQ] stage1_dwconv: convolução com groups=128 (verificar suporte no hls4ml)
[ConversorHGQ] stage2_dwconv: convolução com groups=256 (verificar suporte no hls4ml)
[ConversorHGQ] stage3_dwconv: convolução com groups=640 (verificar suporte no hls4ml)
[ConversorHGQ] stage4_dwconv: convolução com groups=1024 (verificar suporte no hls4ml)
[ConversorHGQ] stage1_dwconv: convolução com groups=128 (verificar suporte no hls4ml)
[ConversorHGQ] stage2_dwconv: convolução com groups=256 (verificar suporte no hls4ml)
[ConversorHGQ] stage3_dwconv: convolução com groups=640 (verificar suporte no hls4ml)
[ConversorHGQ] stage4_dwconv: convolução com groups=1024 (verificar suporte no hls4ml)


Model: "SegFormerB0_DPU_HGQ"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ x (InputLayer)      │ (None, 128, 128,  │          0 │ -                 │
│                     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed1_proj_… │ (None, 134, 134,  │          0 │ x[0][0]           │
│ (ZeroPadding2D)     │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed1_proj   │ (None, 32, 32,    │     25,230 │ patch_embed1_pro… │
│ (QConv2D)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed1_norm   │ (None, 32, 32,    │        418 │ patch_embed1_pro… │
│ (QBatchNormalizati… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_fc1          │ (None, 32, 32,    │     16,994 │ patch_embed1_nor… │
│ (QConv2D)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_dwconv       │ (None, 32, 32,    │      5,506 │ stage1_fc1[0][0]  │
│ (QConv2D)           │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_act          │ (None, 32, 32,    │          0 │ stage1_dwconv[0]… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_fc2          │ (None, 32, 32,    │     16,898 │ stage1_act[0][0]  │
│ (QConv2D)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed2_proj_… │ (None, 34, 34,    │          0 │ stage1_fc2[0][0]  │
│ (ZeroPadding2D)     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed2_proj   │ (None, 16, 16,    │     74,082 │ patch_embed2_pro… │
│ (QConv2D)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed2_norm   │ (None, 16, 16,    │        834 │ patch_embed2_pro… │
│ (QBatchNormalizati… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_fc1          │ (None, 16, 16,    │     66,754 │ patch_embed2_nor… │
│ (QConv2D)           │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_dwconv       │ (None, 16, 16,    │     11,010 │ stage2_fc1[0][0]  │
│ (QConv2D)           │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_act          │ (None, 16, 16,    │          0 │ stage2_dwconv[0]… │
│ (Activation)        │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage2_fc2          │ (None, 16, 16,    │     66,562 │ stage2_act[0][0]  │
│ (QConv2D)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed3_proj_… │ (None, 18, 18,    │          0 │ stage2_fc2[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed3_proj   │ (None, 8, 8, 160) │    369,474 │ patch_embed3_pro

 Total params: 6,724,422 (20.84 MB)

 Trainable params: 5,040,267 (19.23 MB)

 Non-trainable params: 1,684,155 (1.61 MB)

... (110 outputs left out)

Época 41 [Validação]: 100%|██████████| 129/129 [00:17<00:00,  7.57it/s]

Resumo da Época 41 -> Loss: 0.7805 | F1: 0.6392
1456/1456 ━━━━━━━━━━━━━━━━━━━━ 558s 382ms/step - loss: 0.7805 - beta: 1.0000e-11 - ebops: 11066845740.0000
Epoch 42/200


1222/1456 ━━━━━━━━━━━━━━━━━━━━ 1:26 369ms/step - loss: 0.7629

KeyboardInterrupt: 

Testar mag1c + RGB

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_" + selected_loss + "_mag1c_rgb"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"])


Testar apenas mag1c

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_mag1c"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c"])


Testar Varon

In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_varon"
    avaliar_e_visualizar(MODELO, nome_modelo, ["ratio_wv3_B7_B5_varon21_sum_c_out", "ratio_wv3_B8_B5_varon21_sum_c_out", "ratio_wv3_B7_B6_varon21_sum_c_out"])


Testar todas as bandas + mag1c


In [ ]:
for MODELO, NOME_MODELO, _ in MODELS:
    nome_modelo = NOME_MODELO + "_all_bands"
    avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", 
                                                "TOA_AVIRIS_640nm", "TOA_AVIRIS_2004nm", "TOA_AVIRIS_2109nm",
                                                "TOA_AVIRIS_2310nm", "TOA_AVIRIS_2350nm", "TOA_AVIRIS_2360nm"])

In [ ]:
import torch

from thop import profile


def calcular_gmacs(modelo, device, canais_entrada=4, altura=512, largura=512):
    modelo.eval()
    
    # Cria um tensor vazio (dummy) com a resolução que você quer testar
    # Usamos Batch Size = 1 para padronizar o custo por imagem
    tensor_teste = torch.randn(1, canais_entrada, altura, largura).to(device)
    
    # O thop rastreia o forward pass e conta as operações
    macs, params = profile(modelo, inputs=(tensor_teste, ), verbose=False)
    
    # 1 GMAC = 1 bilhão de MACs
    gmacs = macs / 1e9
    
    print(f"\n--- Análise de Custo Computacional ---")
    print(f"Resolução de Entrada : {altura}x{largura} ({canais_entrada} canais)")
    print(f"Parâmetros Totais    : {params / 1e6:.2f} M")
    print(f"Custo Computacional  : {gmacs:.3f} GMACs")
    
    return gmacs

# Defina o dispositivo e a quantidade de canais do seu dataset atual (mag1c + 3 RGB = 4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
canais_entrada = 4

print("=== Relatório de Custos Computacionais (Inferência) ===")

for ModeloClass, nome_modelo, _ in MODELS:
    print(f"\nAnalisando arquitetura: {nome_modelo}")
    
    # 1. Instancia o modelo do zero na GPU
    modelo_instanciado = ModeloClass(in_channels=canais_entrada, out_channels=1).to(device)
    
    # 2. Calcula os GMACs simulando a resolução final de teste (512x512)
    gmacs_512 = calcular_gmacs(
        modelo=modelo_instanciado, 
        device=device, 
        canais_entrada=canais_entrada, 
        altura=512, 
        largura=512
    )